# **Laboratorium. Wyjątki i testy jednostkowe w Pythonie**



# Zadanie podstawowe — Walidator danych użytkownika

Napisz program, który waliduje dane użytkownika rejestrującego się w systemie. Program powinien sprawdzać poprawność imienia, wieku oraz adresu e-mail. W przypadku błędnych danych powinien zgłaszać odpowiednie wyjątki.

**Wymagania: **

1. utwórz funkcję validate_user(name, age, email),
2. sprawdź, czy name nie jest pustym napisem,
3. sprawdź, czy age jest liczbą całkowitą większą lub równą 18,
4. sprawdź, czy email zawiera znak @ oraz kropkę po znaku @,
dla błędnych danych zgłaszaj wyjątek ValueError,
5. napisz testy jednostkowe sprawdzające:

* poprawne dane,
* puste imię,
* zbyt niski wiek,
* niepoprawny typ wieku,
* błędny adres e-mail.

Z czego należy skorzystać:

* raise,
* try/except,
* ValueError,
* pytest lub unittest,
* testowanie wyjątków, np. pytest.raises().

In [ ]:
#!pip install -q pytest

In [ ]:
%%writefile test_parametrize.py

def validate_name(name):
    if not isinstance(name, str):
        raise ValueError("Imie musi byc typu string")
    if name == "":
        raise ValueError("Imie nie moze byc puste")
    return True
    
def validate_age(age):
    if not isinstance(age, int):
        raise ValueError("Wiek musi byc liczba calkowita")
    if age < 18:
        raise ValueError("Wiek nie moze byc mniejszy od 18")
    return True

def validate_email(email):
    if not isinstance(email, str):
        raise ValueError("Email musi byc typu string")
    if "@" not in email:
        raise ValueError("Email musi zawierac @")
    if "." not in email.split("@")[-1]:
        raise ValueError("Email po znaku @ musi zawierac znak .")
    return True

def validate_user(name, age, email):
    validate_name(name) 
    validate_age(age) 
    validate_email(email)
    
    return True 




import pytest

@pytest.mark.parametrize(
        "name, raise_exception",
        [
            ("Czarek", False),
            ("", True),
            (18, True),
            ("Kamil", False)
        ]
)
def test_name(name, raise_exception):
    if raise_exception:
        with pytest.raises(ValueError):
            validate_name(name)
    else:
        assert validate_name(name) is True

@pytest.mark.parametrize(
        "age, raise_exception",
        [
            ("Czarek", True),
            (17, True),
            (18, False),
            (58, False),
            (56.3, True)
        ]
)
def test_name(age, raise_exception):
    if raise_exception:
        with pytest.raises(ValueError):
            validate_age(age)
    else:
        assert validate_age(age) is True

@pytest.mark.parametrize(
        "email, raise_exception",
        [
            ("Czarek", True),
            (17, True),
            ("Czarek@AGH.pl", False),
            ("Czarek.AGH.pl", True),
            ("Czarek@AGHpl", True)
        ]
)
def test_name(email, raise_exception):
    if raise_exception:
        with pytest.raises(ValueError):
            validate_email(email)
    else:
        assert validate_email(email) is True


In [ ]:
!pytest -q test_parametrize.py

# Zadanie średniozaawansowane — System rezerwacji miejsc

Zaprojektuj prosty system rezerwacji miejsc na wydarzenie. Program powinien pozwalać na rezerwowanie miejsc, anulowanie rezerwacji oraz sprawdzanie dostępności. Należy zastosować własne wyjątki i przetestować różne scenariusze działania systemu.

**Wymagania:**

1. utwórz klasę ReservationSystem,
2. konstruktor powinien przyjmować liczbę dostępnych miejsc,
3. zaimplementuj metody:

`reserve(user_id)` — rezerwuje miejsce dla użytkownika,
`cancel(user_id)` — anuluje rezerwację użytkownika,
`available_seats()` — zwraca liczbę wolnych miejsc,
`is_reserved(user_id)` — sprawdza, czy użytkownik ma rezerwację,

4. utwórz własne wyjątki:

`NoSeatsAvailableError,`
`AlreadyReservedError,`
`ReservationNotFoundError,`

5. program powinien uniemożliwiać:
* rezerwację, gdy nie ma miejsc,
* ponowną rezerwację przez tego samego użytkownika,
* anulowanie rezerwacji, która nie istnieje,

6. napisz testy jednostkowe dla wszystkich poprawnych i błędnych przypadków.

Z czego należy skorzystać:

* klasy i obiekty,
* własne klasy wyjątków,
* raise,
* testowanie metod klasy,
* asercje,
* testowanie wyjątków,
* przygotowanie danych testowych.

In [6]:
%%writefile reservation.py
class NoSeatsAvailableError(Exception):
    pass

class AlreadyReservedError(Exception):
    pass

class ReservationNotFoundError(Exception):
    pass


class ReservationSystem:
    def __init__(self, seats_num):
        self.seats_num = seats_num
        self.reservation = []

    def reserve(self, user_id):
        if user_id in self.reservation:
            raise AlreadyReservedError("Juz posiadasz rezerwacje")
        elif len(self.reservation) == self.seats_num:
            raise NoSeatsAvailableError("Nie ma wolnych miejsc")
        else:
            self.reservation.append(user_id)

    def cancel(self, user_id):
        if user_id not in self.reservation:
            raise ReservationNotFoundError("Brak rezerwacji")
        else:
            self.reservation.remove(user_id)

    def available_seats(self):
        return self.seats_num - len(self.reservation)

    def is_reserved(self, user_id):
        return user_id in self.reservation

    

import pytest

def test_reservation():
    R = ReservationSystem(5)
    R.reserve(1)
    R.reserve(2)
    assert R.available_seats() == 3

def test_unavailable_reservation():
    R = ReservationSystem(1)
    R.reserve(1)
    with pytest.raises(NoSeatsAvailableError):
        R.reserve(2)

def test_user_already_reserved_seat():
    R = ReservationSystem(10)
    R.reserve(1)
    with pytest.raises(AlreadyReservedError):
        R.reserve(1)

def test_cancel_reservation_with_reservation():
    R = ReservationSystem(5)
    R.reserve(1)
    R.reserve(2)
    R.cancel(1)
    assert R.available_seats() == 4

def test_cancel_reservation_without_reservation():
    R = ReservationSystem(5)
    with pytest.raises(ReservationNotFoundError):
        R.cancel(1)

def test_seats_number_counter():
    R = ReservationSystem(5)
    R.reserve(1)
    R.reserve(2)
    assert R.available_seats() == 3

def test_reservation_is_registered():
    R = ReservationSystem(5)
    R.reserve(1)
    R.reserve(2)
    R.reserve(3)
    assert R.is_reserved(1) == True
    assert R.is_reserved(4) == False
    

Overwriting reservation.py


In [7]:
!pytest -q reservation.py

.......                                                                  [100%]
7 passed in 0.03s


# **Zadanie zaawansowane — Import i walidacja danych z pliku CSV**

Napisz moduł odpowiedzialny za wczytywanie danych studentów z pliku CSV. Program powinien wykrywać błędy w strukturze pliku, brakujące dane oraz niepoprawne wartości. Należy zaprojektować własne wyjątki oraz przygotować kompleksowy zestaw testów jednostkowych.

**Przykładowy plik CSV:**


```
id,name,grade
1,Anna,4.5
2,Jan,3.0
3,Ewa,5.0
```
**Wymagania:**

1. utwórz klasę StudentRecord,
2. utwórz funkcję `load_students(path)`, funkcja powinna zwracać listę obiektów `StudentRecord`, każdy rekord powinien zawierać:

* id,
* name,
* grade,

3. sprawdź, czy plik istnieje,
4. sprawdź, czy plik ma wymagane kolumny: id, name, grade,
5. sprawdź, czy:

* id jest liczbą całkowitą dodatnią,
* name nie jest pusty,
* grade jest liczbą z zakresu od 2.0 do 5.0,

6. utwórz własne wyjątki:

`FileStructureError,`
`InvalidStudentDataError`,
`EmptyFileError`,

7. obsłuż sytuacje:

* brak pliku,
* pusty plik,
* brak wymaganej kolumny,
* błędny typ danych,
* ocena poza zakresem,
* pusty rekord,

8. napisz testy jednostkowe z wykorzystaniem plików tymczasowych.

**Z czego należy skorzystać:**

* obsługa plików,
* moduł csv,
* własne wyjątki,
* walidacja danych,
* pytest,
* tmp_path w testach,
* testowanie poprawnych i niepoprawnych danych wejściowych.


In [2]:
%%writefile test_student.py
from pathlib import Path
import pytest
import csv

class FileStructureError(Exception):
    pass

class InvalidStudentDataError(Exception):
    pass

class EmptyFileError(Exception):
    pass

class StudentRecord:
    def __init__(self, id, name, grade):
        self.id = id
        self.name = name
        self.grade = grade

    def student_info(self):
        print(f"-----Student info----- \nID => {self.id}\nName => {self.name}\nGrade => {self.grade}")

def validate_id(id_val):
    if not isinstance(id_val, int):
        raise InvalidStudentDataError("ID musi byc typu int")
    if id_val < 1: 
        raise InvalidStudentDataError("ID musi byc dodatanie")
    return True

def validate_name(name):
    if not isinstance(name, str):
        raise InvalidStudentDataError("Imie musi byc typu string")
    if name.strip() == "":
        raise InvalidStudentDataError("Imie nie moze byc puste")
    return True

def validate_grade(grade_val):
    if not isinstance(grade_val, float):
        raise InvalidStudentDataError("Ocena musi byc typu float")
    if not (2.0 <= grade_val <= 5.0):
        raise InvalidStudentDataError("Ocena musi byc z przedzialu od 2.0 do 5.0")
    return True

def validate_student(id_val, name, grade_val):
    return validate_id(id_val) and validate_name(name) and validate_grade(grade_val)

def load_students(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError("Plik nie istnieje")

    student_list = []
    
    with open(path, mode='r', encoding='utf-8') as file:
        info = csv.reader(file)
        
        try:
            head = next(info) 
        except StopIteration:
            raise EmptyFileError("Plik jest pusty.")
        
        for line in info:
            if len(line) != 3:
                raise FileStructureError("Brakujace dane w pliku csv")
            
            try:
                converted_id = int(line[0])
                converted_grade = float(line[2])
                name = line[1]
            except ValueError:
                raise InvalidStudentDataError("Niepoprawny format typu danych w CSV")

            if validate_student(converted_id, name, converted_grade):
                student = StudentRecord(converted_id, name, converted_grade)
                student_list.append(student)
                
    return student_list




def test_load_students_correct_data(tmp_path):
    file_path = tmp_path / "students_correct.csv"
    file_path.write_text("id,name,grade\n1,Jan Kowalski,4.5\n2,Anna Nowak,5.0", encoding="utf-8")
    
    students = load_students(file_path)
    
    assert len(students) == 2
    assert students[0].id == 1
    assert students[0].name == "Jan Kowalski"
    assert students[0].grade == 4.5



def test_load_students_file_not_found():
    with pytest.raises(FileNotFoundError):
        load_students("xyz.csv")


def test_load_students_empty_file(tmp_path):
    file_path = tmp_path / "empty.csv"
    file_path.write_text("", encoding="utf-8") 
    
    with pytest.raises(EmptyFileError):
        load_students(file_path)


def test_load_students_missing_column(tmp_path):
    file_path = tmp_path / "missing_column.csv"
    file_path.write_text("id,name,grade\n1,Jan Kowalski", encoding="utf-8")
    
    with pytest.raises(FileStructureError):
        load_students(file_path)


def test_load_students_invalid_data_type(tmp_path):
    file_path = tmp_path / "invalid_type.csv"
    file_path.write_text("id,name,grade\nXYZ,Jan Kowalski,4.5", encoding="utf-8")
    
    with pytest.raises(InvalidStudentDataError):
        load_students(file_path)


def test_load_students_grade_out_of_range(tmp_path):
    file_path = tmp_path / "grade_out_of_range.csv"
    file_path.write_text("id,name,grade\n1,Jan Kowalski,6.0", encoding="utf-8")
    
    with pytest.raises(InvalidStudentDataError):
        load_students(file_path)


def test_load_students_empty_record(tmp_path):
    file_path = tmp_path / "empty_record.csv"
    file_path.write_text("id,name,grade\n,,", encoding="utf-8")
    
    with pytest.raises(FileStructureError):
        load_students(file_path)

Writing test_student.py


In [3]:
!pytest -q reservation.py

.......                                                                  [100%]
7 passed in 0.02s


# **Zadanie dodatkowe — Testy parametryzowane**

Rozszerz dowolne wcześniejsze zadanie o testy parametryzowane. Celem jest sprawdzenie wielu przypadków testowych bez powielania kodu testów.

**Wymagania:**

1. użyj `pytest.mark.parametrize`,
2. przygotuj minimum 5 przypadków poprawnych,
3. przygotuj minimum 5 przypadków błędnych,
4. sprawdź zarówno poprawny wynik, jak i zgłaszane wyjątki.

**Z czego należy skorzystać:**

* pytest.mark.parametrize,
* testowanie wielu danych wejściowych,
* pytest.raises,
* czytelna organizacja testów.

## Odpowiedź do zadania

W podstawowej wersji zadania wykorzystałem `pytest.mark.parametrize` dla 5 przypadków poprawnych (`raise_exception` ustawione na False) oraz 8 przypadków błędnych (`raise_exception` ustawione na True), gdzie oczekuje wyrzucenie wyjątku



```python
@pytest.mark.parametrize(
        "name, raise_exception",
        [
            ("Czarek", False),
            ("", True),
            (18, True),
            ("Kamil", False)
        ]
)
def test_name(name, raise_exception):
    if raise_exception:
        with pytest.raises(ValueError):
            validate_name(name)
    else:
        assert validate_name(name) is True

@pytest.mark.parametrize(
        "age, raise_exception",
        [
            ("Czarek", True),
            (17, True),
            (18, False),
            (58, False),
            (56.3, True)
        ]
)
def test_name(age, raise_exception):
    if raise_exception:
        with pytest.raises(ValueError):
            validate_age(age)
    else:
        assert validate_age(age) is True

@pytest.mark.parametrize(
        "email, raise_exception",
        [
            ("Czarek", True),
            (17, True),
            ("Czarek@AGH.pl", False),
            ("Czarek.AGH.pl", True),
            ("Czarek@AGHpl", True)
        ]
)
def test_name(email, raise_exception):
    if raise_exception:
        with pytest.raises(ValueError):
            validate_email(email)
    else:
        assert validate_email(email) is True
```